In [2]:
import os

# Find where you are right now
print("Current directory:", os.getcwd())

# List everything in current folder
print("\nFiles here:")
for item in os.listdir('.'):
    print(item)

Current directory: /Users/pallavi_chandanshive/projects/clinical-summarization-eval/notebooks

Files here:
01_data_exploration.ipynb
.ipynb_checkpoints


In [3]:
# Search for CSV files anywhere in your project
for root, dirs, files in os.walk('.'):
    # Skip venv folder
    if 'venv' in root:
        continue
    for file in files:
        if file.endswith('.csv'):
            print(os.path.join(root, file))

In [1]:
import pandas as pd

# Update these paths to match where your files are
notes = pd.read_csv('../data/raw/clinical_notes.csv')
patients = pd.read_csv('../data/raw/patients.csv')
admissions = pd.read_csv('../data/raw/admissions.csv')

In [2]:
# Basic shape
print("=== NOTES ===")
print(notes.shape)
print(notes.columns.tolist())
print(notes.head(2))

print("\n=== PATIENTS ===")
print(patients.shape)
print(patients.columns.tolist())

print("\n=== ADMISSIONS ===")
print(admissions.shape)
print(admissions.columns.tolist())

=== NOTES ===
(1602, 9)
['ingest_timestamp', 'clinical_note_id', 'clean_note_text', 'creation_timestamp', 'updt_dt_tm', 'note_subject', 'note_type', 'admission_id', 'person_id']
   ingest_timestamp                      clinical_note_id  \
0  07/01/2026 14:35  17bf845b-88f8-4604-8983-6e74453aada5   
1  07/01/2026 14:50  e5d9c0a4-299a-425e-abbc-27fabe9cb742   

                                     clean_note_text creation_timestamp  \
0  Patient Name: Judith Ada Wells\n- Patient ID: ...   07/01/2026 14:05   
1  Patient reviewed at 14:20 on 07/01/26 by Nurse...   07/01/2026 14:20   

         updt_dt_tm         note_subject note_type  \
0  07/01/2026 14:35            ED Triage        ED   
1  07/01/2026 14:50  ED Triage Follow-Up        ED   

                           admission_id                             person_id  
0  63720303-3c1b-4356-befd-eea5438da62e  28570119-9cdc-4120-98c0-4edb76cf36a3  
1  63720303-3c1b-4356-befd-eea5438da62e  28570119-9cdc-4120-98c0-4edb76cf36a3  

=== PATI

In [3]:
print("\n=== NOTE TYPES ===")
print(notes['note_type'].value_counts())


=== NOTE TYPES ===
note_type
Orthopaedics Inpatients                      263
Physiotherapy Documentation                  262
Medicine Inpatients                          242
ED                                           155
Neurology Inpatients                         104
Therapies Inpatients                          81
Paediatrics Inpatients                        66
Anaesthetic Documentation                     62
Respiratory Inpatients                        62
ED Depart Summary                             39
Respiratory Medicine Inpatients               37
Pre-op Checklist                              35
Pre-op Consent                                31
Theatre notes                                 31
Neurosurgery Inpatients                       26
Orthopaedics Outpatients                      24
Occupational Therapy Documentation            24
Surgery Inpatients                            21
Dietetics Documentation                       14
Speech and Language Therapy Documentati

In [4]:
# How many notes per admission?
notes_per_admission = notes.groupby('admission_id')['clinical_note_id'].count()
print("\n=== NOTES PER ADMISSION ===")
print(notes_per_admission.describe())
print(notes_per_admission.value_counts().head(10))

# How many admissions per patient?
admissions_per_patient = admissions.groupby('patient_id')['admission_id'].count()
print("\n=== ADMISSIONS PER PATIENT ===")
print(admissions_per_patient.describe())
print(admissions_per_patient.value_counts().head(10))


=== NOTES PER ADMISSION ===
count    69.000000
mean     23.217391
std       8.709139
min      10.000000
25%      16.000000
50%      22.000000
75%      30.000000
max      45.000000
Name: clinical_note_id, dtype: float64
clinical_note_id
22    5
25    5
33    4
17    4
23    4
19    4
35    4
15    4
12    3
26    3
Name: count, dtype: int64

=== ADMISSIONS PER PATIENT ===
count    50.000000
mean      1.380000
std       0.490314
min       1.000000
25%       1.000000
50%       1.000000
75%       2.000000
max       2.000000
Name: admission_id, dtype: float64
admission_id
1    31
2    19
Name: count, dtype: int64


In [5]:
# Pick first patient and show all their notes in order
first_patient = notes['person_id'].iloc[0]
patient_notes = notes[notes['person_id'] == first_patient].sort_values('creation_timestamp')

print(f"\n=== FULL JOURNEY FOR PATIENT {first_patient} ===")
for _, row in patient_notes.iterrows():
    print(f"\nDate: {row['creation_timestamp']}")
    print(f"Type: {row['note_type']}")
    print(f"Text preview: {str(row['clean_note_text'])[:200]}")
    print("---")


=== FULL JOURNEY FOR PATIENT 28570119-9cdc-4120-98c0-4edb76cf36a3 ===

Date: 07/01/2026 14:05
Type: ED
Text preview: Patient Name: Judith Ada Wells
- Patient ID: 28570119-9cdc-4120-98c0-4edb76cf36a3
- NHS Number: 272733208
- Date of Birth: 15/05/84 (39 years old)
- Gender: Female
- Allergies: NKA
- Current Medicatio
---

Date: 07/01/2026 14:20
Type: ED
Text preview: Patient reviewed at 14:20 on 07/01/26 by Nurse Chukwuebuka Okafor. Patient presented with a severe headache rated 8/10 in intensity. BP measured at 160/90 mmHg. HR recorded at 88 bpm. Brief neurologic
---

Date: 07/01/2026 14:45
Type: ED
Text preview: - Patient: Judith Ad a Wells, 39-year-old female, DOB: 15/05/84, NHS Number: 272733208.
 - Date/Time: 07/01/26, 14:45.
 - Staff involved: Nurse Jasmine Freda Murray.
 - Chief Complaint: Severe headach
---

Date: 07/01/2026 15:15
Type: ED
Text preview: Patient: Judith Ada Wells, 39-yer-old female, presenting with severe headache rated 8/10 in intensity after exertion. Current 

In [6]:
# Check duration of admissions
notes['creation_timestamp'] = pd.to_datetime(
    notes['creation_timestamp'], 
    format='%d/%m/%Y %H:%M',
    dayfirst=True
)

date_range = notes.groupby('admission_id')['creation_timestamp'].agg(['min', 'max'])
date_range['duration_days'] = (date_range['max'] - date_range['min']).dt.days

print("=== ADMISSION DURATION ===")
print(date_range['duration_days'].describe())

# Check patients with 2 admissions
admissions_per_patient = admissions.groupby('patient_id')['admission_id'].count()
two_admission_patients = admissions_per_patient[admissions_per_patient == 2].index.tolist()
print(f"\nPatients with 2 admissions: {len(two_admission_patients)}")

# Check note types for multi-admission patients
multi_notes = notes[notes['person_id'].isin(two_admission_patients)]
print(f"Total notes for multi-admission patients: {len(multi_notes)}")

# Find discharge summary note type
print("\n=== DISCHARGE SUMMARY NOTES ===")
discharge_notes = notes[notes['note_type'].str.contains('Depart|Discharge|Summary', 
                                                          case=False, na=False)]
print(f"Count: {len(discharge_notes)}")
print(discharge_notes['note_type'].value_counts())

=== ADMISSION DURATION ===
count    69.000000
mean      8.869565
std       6.216470
min       0.000000
25%       4.000000
50%       7.000000
75%      15.000000
max      20.000000
Name: duration_days, dtype: float64

Patients with 2 admissions: 19
Total notes for multi-admission patients: 988

=== DISCHARGE SUMMARY NOTES ===
Count: 39
note_type
ED Depart Summary    39
Name: count, dtype: int64
